# 🧠 Assistant LLM basé sur le RAG — spécialisé **DevOps / MLOps**

**TP — Développement d'un assistant intelligent (Retrieval-Augmented Generation)**

Cet assistant répond à des questions en s'appuyant sur un corpus de cours DevOps & MLOps
(PDF, Markdown, notes). Il propose **deux modes**, et leur contraste *est* la leçon MLOps :

- **MODE LLM** — *intuition* : le modèle répond avec ses seules connaissances pré-entraînées,
  sans source (réponse non ancrée).
- **MODE RAG** — *preuve* : le même modèle, mais la réponse est **fondée sur le corpus** et
  porte des **citations [n]** vérifiables.

### Architecture RAG

```
Document(s)  ─►  Nettoyage + Chunking  ─►  Embeddings (SentenceTransformers)
                                                    │
                                                    ▼
Question  ─►  Embedding requête  ─►  Recherche FAISS (top-k)  ─►  Contexte
                                                    │
                                                    ▼
                              Prompt (système + contexte + question)
                                                    │
                                                    ▼
                           LLM open-source (Transformers)  ─►  Réponse fondée [n]
```

### Pile technique
| Rôle | Outil |
|------|-------|
| Gestion de projet / env | **uv** |
| Extraction PDF | **PyMuPDF** |
| Embeddings | **SentenceTransformers** (`multilingual-e5-small`) |
| Index & recherche sémantique | **FAISS** |
| Génération | **Transformers** (`Qwen2.5-1.5B-Instruct`) |
| API de démonstration | **FastAPI** |
| Interface web | **Astro** + **Three.js** |

> Le code réutilisable est factorisé dans le paquet `rag_devops` (dossier `src/`).

## 0. Préparation de l'environnement

Dépendances installées via **uv** (`uv sync --extra notebook`). Noyau du projet : `uv run jupyter lab`.

In [1]:
import sys, os, textwrap
sys.path.insert(0, os.path.abspath("../src"))  # rend src/ importable

import torch
from rag_devops.config import settings

print("Python :", sys.version.split()[0])
print("PyTorch :", torch.__version__)
print("CUDA :", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("\n--- Configuration ---")
print("Corpus      :", settings.corpus_dir)
print("Embeddings  :", settings.embedding_model)
print("LLM         :", settings.llm_model)
print("Device      :", settings.resolve_device())
print("chunk_size  :", settings.chunk_size, "| overlap :", settings.chunk_overlap, "| top_k :", settings.top_k)

Python : 3.12.13
PyTorch : 2.15.0+cu132
CUDA : True | NVIDIA GeForce RTX 4060 Laptop GPU

--- Configuration ---
Corpus      : /home/magreyuptown/Downloads/Devops - Lake
Embeddings  : intfloat/multilingual-e5-small
LLM         : Qwen/Qwen2.5-1.5B-Instruct
Device      : cuda
chunk_size  : 1000 | overlap : 180 | top_k : 5


## 1. Préparation et nettoyage du corpus

**Objectif :** charger les documents, extraire le texte, le nettoyer, puis le découper en
*chunks* adaptés à la recherche sémantique.

- **Chargement** : PDF (PyMuPDF), Markdown, `.txt`.
- **Nettoyage** : recollage des mots coupés, suppression des numéros de page, normalisation.
- **Chunking** : agrégation par paragraphes (~1000 caractères) avec **recouvrement** (~180)
  pour préserver le contexte aux frontières.

In [2]:
from rag_devops.ingest import build_corpus, discover_files

files = discover_files(settings.corpus_dir)
print(f"{len(files)} fichiers détectés.\n")
chunks = build_corpus(settings, verbose=False)
print(f"→ {len(chunks)} chunks générés.")

121 fichiers détectés.



/home/magreyuptown/Projects/LLM-RAG/.venv/lib/python3.12/site-packages/fitz/__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(


→ 2868 chunks générés.


In [3]:
from collections import Counter
import statistics

rep = Counter(c.category for c in chunks)
print("Répartition des chunks par thème :")
for cat, n in rep.most_common():
    print(f"  {cat:<16} {n:>4}  " + "#" * max(1, round(n / max(rep.values()) * 30)))
longs = [len(c.text) for c in chunks]
print(f"\nTaille des chunks : min={min(longs)}, médiane={int(statistics.median(longs))}, max={max(longs)}")

Répartition des chunks par thème :
  RAW              2300  ##############################
  devops-notes      170  ##
  Mlops             122  ##
  K8's               75  #
  BASIC              52  #
  Docker - CD        39  #
  Exams              38  #
  SONARQUBE          36  #
  Monitoring         32  #
  (racine)            4  #

Taille des chunks : min=80, médiane=999, max=1177


In [4]:
ex = chunks[len(chunks)//2]
print(f"Chunk #{ex.id} | source : {ex.source} | thème : {ex.category}\n" + "-"*70)
print(textwrap.fill(ex.text[:700], width=90))

Chunk #1434 | source : RAW/Site Reliability Engineering How Google Runs Production Systems_Jennifer_Petoff.pdf | thème : RAW
----------------------------------------------------------------------
towards minimizing accidental complexity, SRE teams should: - Push back when accidental
complexity is introduced into the systems for which they are responsible - Constantly
strive to eliminate complexity in systems they onboard and for which they assume
operational responsibility I Won’t Give Up My Code! Because engineers are human beings who
often form an emotional attachment to their creations, confrontations over large-scale
purges of the source tree are not uncommon. Some might protest, “What if we need that code
later?” “Why don’t we just comment the code out so we can easily add it again later?” or
“Why don’t we gate the code with a flag instead of deleting it?” These are all terrible


## 2. Création des embeddings

Un **embedding** transforme un texte en vecteur. Deux textes proches sémantiquement ont des
vecteurs proches (forte **similarité cosinus**) → recherche *par le sens*.

Modèle multilingue **`intfloat/multilingual-e5-small`** (FR + EN). Préfixes `query:`/`passage:`
gérés par `Embedder`. Vecteurs **normalisés L2** ⇒ produit scalaire = cosinus.

In [5]:
from rag_devops.embed import Embedder

embedder = Embedder(settings)
print("Dimension des embeddings :", embedder.dim)

phrases = [
    "Kubernetes orchestre des conteneurs sur un cluster.",       # 0
    "K8s gère le déploiement et la montée en charge des pods.",  # 1  (proche de 0)
    "MLflow permet de suivre les expériences de machine learning.",  # 2 (autre sujet)
]
V = embedder.encode_passages(phrases, show_progress=False)
sim = V @ V.T
print("\nMatrice de similarité cosinus :\n        P0     P1     P2")
for i, row in enumerate(sim):
    print(f"P{i}   " + "  ".join(f"{v:5.3f}" for v in row))
print("\n→ P0 et P1 (K8s) sont bien plus proches que P0 et P2 (MLflow).")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Dimension des embeddings : 384

Matrice de similarité cosinus :
        P0     P1     P2
P0   1.000  0.882  0.867
P1   0.882  1.000  0.872
P2   0.867  0.872  1.000

→ P0 et P1 (K8s) sont bien plus proches que P0 et P2 (MLflow).


/home/magreyuptown/Projects/LLM-RAG/src/rag_devops/embed.py:37: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  return self.model.get_sentence_embedding_dimension()


## 3. Recherche sémantique avec FAISS

**FAISS** retrouve rapidement les plus proches voisins. `IndexFlatIP` + vecteurs normalisés
⇒ similarité **cosinus** exacte. On encode tous les chunks, on construit l'index, puis on le
**persiste** (`data/index/`) pour le recharger sans tout recalculer (API FastAPI).

In [6]:
from rag_devops.index import VectorStore

store = VectorStore(settings, embedder=embedder)
store.build(chunks=chunks, verbose=True)
store.save()
print("\nIndex persisté dans :", settings.index_dir)

🧠 Calcul des embeddings (intfloat/multilingual-e5-small)…


Batches:   0%|          | 0/45 [00:00<?, ?it/s]

✅ Index FAISS prêt : 2868 vecteurs (dim=384)

Index persisté dans : /home/magreyuptown/Projects/LLM-RAG/data/index


In [7]:
def montrer_recherche(q, k=3):
    print(f"Requête : {q}\n" + "="*80)
    for i, hit in enumerate(store.search(q, top_k=k), 1):
        c = hit.chunk
        print(f"[{i}] score={hit.score:.3f} | {c['source']} ({c['category']})")
        print("    " + textwrap.shorten(c['text'].replace(chr(10), ' '), width=180) + "\n")

montrer_recherche("Comment fonctionne un Pod dans Kubernetes ?")
montrer_recherche("À quoi sert MLflow ?")

Requête : Comment fonctionne un Pod dans Kubernetes ?
[1] score=0.896 | K8's/Chapitre 5 - Kubernetes & Orchestration - Concepts avancés.pdf (K8's)
    Pod - La plus petite unité Kubernetes © 2026 - 2027 – ESPRIT – UP ASI – DevOps Définition Un Pod est la plus petite unité déployable dans Kubernetes. Il encapsule un ou [...]

[2] score=0.895 | K8's/Chapitre 5 - Kubernetes & Orchestration - Concepts avancés.pdf (K8's)
    présent sur chaque nœud - Écoute l'API Server pour connaître les Pods assignés - Lance les conteneurs et surveille leur état - Container Runtime : Logiciel chargé de [...]

[3] score=0.894 | Exams/Examen_DevOps.pdf (Exams)
    ds disponibles en fonction des ressources. C. Gère la communication réseau et la redirection du trafic vers les Pods. D. Exécute et surveille les Pods sur chaque nœud du [...]

Requête : À quoi sert MLflow ?
[1] score=0.911 | Mlops/MLflow - versioning/3- MLflow.pdf (Mlops)
    - MLflow est une plateforme open source développé par databricks en 201

## 4. Génération avec un LLM open-source — les DEUX modes

On intègre **`Qwen/Qwen2.5-1.5B-Instruct`** via Transformers. Le prompt système spécialise le
modèle en *professeur DevOps/MLOps*. On compare les deux modes sur la **même question** —
c'est la leçon centrale du MLOps.

In [8]:
from rag_devops.llm import LLM
from rag_devops.pipeline import RAGPipeline, RAG_SYSTEM_PROMPT

print("Prompt système (mode RAG, citations [n] imposées) :\n" + "-"*70)
print(textwrap.fill(RAG_SYSTEM_PROMPT, width=90))

llm = LLM(settings).load()
rag = RAGPipeline(store, llm, settings)
print("\nLLM chargé sur :", llm.device)

Prompt système (mode RAG, citations [n] imposées) :
----------------------------------------------------------------------
Tu es un professeur expert en DevOps et MLOps. Tu enseignes de façon claire et structurée,
en français.  RÈGLES IMPÉRATIVES : 1. Réponds UNIQUEMENT à partir du CONTEXTE fourni
(extraits numérotés du support de cours). N'invente jamais d'information absente du
contexte. 2. Appuie CHAQUE affirmation sur une citation au format [n] correspondant au
numéro de l'extrait utilisé (ex. « Un Pod regroupe un ou plusieurs conteneurs [2]. »). 3.
Si le contexte ne contient pas la réponse, dis-le : « Le support de cours ne couvre pas ce
point. » et n'invente rien. 4. Structure : définition, explication, exemple ou commande si
pertinent.


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]


LLM chargé sur : cuda


In [9]:
# Prompt réellement envoyé au modèle en mode RAG (contexte numéroté)
q = "C'est quoi l'intégration continue (CI) ?"
hits = store.search(q, top_k=settings.top_k)
sys_p, user_p = rag.build_prompt(q, hits)
print(textwrap.fill(user_p[:1100], width=95))
print("\n... [contexte tronqué] ...")

CONTEXTE (extraits numérotés du support de cours) : [1] (source: RAW/What is CI.pdf · thème:
RAW) Qu’est-ce que l’Intégration Continue (CI) ? Qu’est-ce que l’intégration continue (CI) ?
L'intégration continue, (ou Continuous Integration CI), est une pratique DevOps qui consiste à
régulièrement intégrer les modifications faites dans un dépôt de code. - Le principe consiste à
automatiser les tâches habituellement effectuées à la main, telles que le build de
l'application et l’exécution des tests. - L’objectif est d’automatiser ces étapes afin d'éviter
les erreurs humaines et de pouvoir vérifier la validité d’une application au fur et à mesure
des évolutions du code. L'intégration continue est la première phase dans le mouvement DevOps
qui permet d'améliorer la collaboration et la communication entre les équipes de développement
(Dev) et d'exploitation (Ops) dans le but de livrer des logiciels de bonne qualité. Cette
première phase correspond à une série d’étapes (ou pipeline CI) qui doiv

### 🔀 Le contraste : MODE LLM (intuition) vs MODE RAG (preuve)

In [10]:
question = "C'est quoi un Pod dans Kubernetes et à quoi sert-il ?"

print("█ MODE LLM — intuition (sans récupération, sans source)\n" + "-"*80)
r_llm = rag.ask(question, mode="llm")
print(textwrap.fill(r_llm.answer, width=95))

print("\n\n█ MODE RAG — preuve (ancré sur le cours, avec citations [n])\n" + "-"*80)
r_rag = rag.ask(question, mode="rag")
print(textwrap.fill(r_rag.answer, width=95))
print("\n— Sources citées —")
for s in r_rag.sources:
    print(f"  [{s['n']}] {s['source']}  (score {s['score']})")

█ MODE LLM — intuition (sans récupération, sans source)
--------------------------------------------------------------------------------


Un Pod dans Kubernetes est comme une boîte où on met plusieurs processus ou services pour
qu'ils travaillent ensemble. Imagine que tu as des amis qui veulent jouer ensemble mais ils
vivent dans différentes pièces du même immeuble. Pour qu'ils puissent jouer ensemble, chaque
ami doit avoir son propre endroit sûr et tranquille.  Dans Kubernetes, les Pods sont comme ces
pièces sécurisées. Ils contiennent tous les éléments nécessaires pour un service fonctionner
correctement, comme l'image d'un programme, ses dépendances, et tout ce dont il a besoin pour
exécuter correctement.   En utilisant des Pods, Kubernetes permet aux différents composants
d'un système informatique de se connecter entre eux, sans avoir besoin de créer des liaisons
manuelles. Cela rend la gestion du système plus simple et efficace, car tout le monde travaille
sur le même projet, comme si chacun était dans la même pièce.


█ MODE RAG — preuve (ancré sur le cours, avec citations [n])
-------------------------------------

Un Pod dans Kubernetes est la plus petite unité déployable, encapsulant un ou plusieurs
conteneurs qui partagent le même réseau et le même stockage. Ce concept est décrit dans le
chapitre 5 du document "K8's" sous le titre "Kubernetes & Orchestration - Concepts avancés".
Un Pod possède une adresse IP unique dans le cluster, permettant ainsi à ces conteneurs de
communiquer entre eux via localhost. De plus, les Pods sont éphémères, signifiant qu'ils
peuvent être supprimés et recréés à tout moment sans avoir besoin de réinitialiser tous les
conteneurs associés. En outre, un Pod ne se recrée pas seul, mais plutôt par l'intervention du
Deployment qui gère cette opération.  En termes d'anatomie, un Pod comprend généralement un ou
plusieurs conteneurs (compris l'application principale et ses sidecars), un volume partagé
entre les conteneurs du Pod, une IP unique dans le namespace réseau du cluster, et des
variables d'environnement injectées via ConfigMap ou Secret. Ces éléments contribuent à 

### Garde-fou anti-hallucination (mode RAG)

Pour une question **hors corpus**, l'assistant ancré doit reconnaître qu'il ne sait pas.

In [11]:
r = rag.ask("Quelle est la capitale de l'Australie ?", mode="rag")
print(textwrap.fill(r.answer, width=95))

Le contexte fournit aucune information relative à la capitale de l'Australie. Les informations
fournies concernent les technologies et concepts liés au développement opérationnel et aux
outils de gestion de projet, mais ne mentionnent pas de données géographiques spécifiques comme
celles relatives à la capitale de l'Australie.


## 5. Démonstration — plusieurs questions (mode RAG)

In [12]:
for q in [
    "Pourquoi utiliser Docker pour conteneuriser une application ?",
    "Quel est le rôle de SonarQube dans un pipeline CI/CD ?",
    "À quoi servent Prometheus et Grafana dans le monitoring ?",
    "Qu'est-ce que MLflow apporte à un projet de machine learning ?",
]:
    r = rag.ask(q, mode="rag")
    print("Q:", q, "\n" + "-"*80)
    print(textwrap.fill(r.answer, width=95))
    print("   sources :", ", ".join(f"[{s['n']}] {s['source']}" for s in r.sources[:3]), "\n")

Q: Pourquoi utiliser Docker pour conteneuriser une application ? 
--------------------------------------------------------------------------------
Un conteneur est un environnement isolé qui contient une application et ses dépendances,
garantissant ainsi une expérience uniforme lors de sa mise en œuvre sur différentes machines.
En utilisant Docker, nous pouvons créer ces conteneurs rapidement et efficacement, facilitant
ainsi le déploiement et la maintenance de nos applications.  Selon le contexte, il est
mentionné que Docker permet de créer des environnements isolés où une application et ses
dépendances peuvent être embarquées, ce qui rend possible leur utilisation partout sans
modification. Cette isolation assure également une sécurité accrue, car elle limite les
interactions entre différents conteneurs.  La question demande pourquoi utiliser Docker pour
conteneuriser une application. Selon le contexte, ceci est effectué pour répondre à plusieurs
problèmes rencontrés lorsque l'on uti

Q: Quel est le rôle de SonarQube dans un pipeline CI/CD ? 
--------------------------------------------------------------------------------
Le rôle de SonarQube dans un pipeline CI/CD est crucial car il intervient après les tests et
avant le déploiement. Son rôle principal est de garantir que le code ait atteint un niveau
suffisamment élevé de qualité avant de passer à la phase de déploiement. SonarQube utilise des
règles et des métriques spécifiques pour analyser le code, identifiant ainsi les bugs, les
vulnérabilités et les code smells. Ces analyses sont essentielles pour assurer que le code est
performant, sécurisé et facile à maintenir. Par conséquent, SonarQube joue un rôle central dans
la gestion de la qualité du code tout au long du pipeline CI/CD, contribuant ainsi à la
conformité et à la fiabilité du produit final.
   sources : [1] SONARQUBE/Atelier 6 - Intégration SonarQube dans un pipeline Jenkins.pdf, [2] SONARQUBE/Chapitre 4 - SonarQube & Qualité du code.pdf, [3] SONARQUBE

Q: À quoi servent Prometheus et Grafana dans le monitoring ? 
--------------------------------------------------------------------------------
À quoi servent Prometheus et Grafana dans le monitoring ?   Prometheus et Grafana jouent un
rôle crucial dans le monitoring de l’application, notamment :  1. **Collecte de métriques** :
Prometheus collecte des informations sur l'état de l'environnement, telles que le nombre de
connexions, les erreurs, etc., en utilisant des scripts de collecte basés sur des langages
spécifiques (PromQL pour Prometheus).  2. **Visualisation des métriques** : Grafana permet de
visualiser ces métriques collectées par Prometheus sous forme de graphiques, tables, etc. Ces
visualisations sont essentielles pour comprendre comment fonctionne l'environnement et
identifier les problèmes potentiels.  3. **Alerting** : Grâce à l'alertmanager, Prometheus peut
envoyer des alertes lorsque certaines conditions sont remplies, indiquant qu'une métrique
atteint un seuil critique o

Q: Qu'est-ce que MLflow apporte à un projet de machine learning ? 
--------------------------------------------------------------------------------
MLflow apporte plusieurs avantages significatifs à un projet de machine learning :  1. **Suivi
et Gestion Structurées** : MLflow facilite le suivi des expériences de machine learning, en
enregistrant les paramètres, métriques et artefacts associés à chaque itération. Cela permet
une analyse précise des performances des modèles et une comparaison facile entre les versions.
2. **Standardisation des Projets** : MLflow structure les projets de machine learning pour
qu'ils soient reproductibles et faciles à partager. Il aide à organiser les fichiers et les
scripts nécessaires à la création, à l'évaluation et au déploiement des modèles.  3.
**Simplification de la Collaboration** : MLflow facilite la collaboration entre Data Scientists
et Développeurs, en fournissant des outils pour la gestion efficace des modèles et leur mise en
production.  4. *

## 6. Bonus — Résumé de document

Produit un **résumé pédagogique structuré** d'un thème du corpus (agrégation des chunks → LLM).

In [13]:
from rag_devops.summarize import summarize_document

resume = summarize_document(store, "FastAPI", llm=llm, cfg=settings)
print("RÉSUMÉ — thème « FastAPI »\n" + "="*80)
print(textwrap.fill(resume, width=95))

RÉSUMÉ — thème « FastAPI »
Résumé pédagogique structuré :  **Sujet :** Création d'une API REST pour effectuer des
prédictions avec un modèle Machine Learning  **Points clés :** - Utilisation de FastAPI pour
développer des API - Performance élevée et intégration facile avec MLOps - Documentation
automatisée via Swagger - Prise en charge des API REST par FastAPI - Installation de FastAPI et
Uvicorn - Sauvegarde et chargement du modèle  **Commandes importantes :** - `source
venv/bin/activate` pour activer l'environnement virtuel - `pip install fastapi uvicorn` -
`python -m uvicorn main:app --reload`  **Résumé :** Ce tutoriel vous aide à créer une API REST
pour effectuer des prédictions avec un modèle Machine Learning. Vous allez installer FastAPI et
Uvicorn, charger un modèle existant, puis utiliser FastAPI pour gérer cette API. Le tutoriel
inclut également comment planifier un atelier future pour exposer la fonctionnalité de
prediction comme service REST.


## 7. Bonus — Interface (API FastAPI + front Astro/Three.js)

- **API FastAPI** (`api/main.py`) — `/ask` (modes `llm` / `rag`), `/search`, `/summarize`, `/info`.
  ```bash
  uv run uvicorn api.main:app --port 8000
  ```
- **Front Astro** (`web/`) — hero Three.js (espace latent → graphe de connaissances), sélecteur
  de mode matériel LLM/RAG, citations `[n]` type Perplexity, composer façon claude.ai.
  ```bash
  cd web && pnpm install && pnpm dev   # http://localhost:4321
  ```

## 8. Conclusion

1. ✅ **Corpus** chargé, nettoyé, découpé en chunks.
2. ✅ **Embeddings** sémantiques (SentenceTransformers).
3. ✅ **Index FAISS** + recherche sémantique pertinente.
4. ✅ **Génération** par LLM open-source, deux modes (intuition vs preuve).
5. ✅ Réponses **ancrées et citées [n]** + garde-fou anti-hallucination.
6. ✅ **Bonus** : résumé + interface web (FastAPI + Astro/Three.js).

**Pistes :** reranking (cross-encoder), découpage sémantique, évaluation (RAGAS), historique conversationnel.